# 1. Generate dataset (each teammate runs this once)

Each person uses a **different `SEED_START`** so nobody generates the same workloads. The result is saved as a CSV in your Google Drive folder.

**Seed blocks (one per person):** p1 = 0, p2 = 10000, p3 = 20000. The final test set uses 900000 with `PER_FAMILY = 100`. Keep `PER_FAMILY` at or below 10000.

**Two dataset versions?** Use a separate `DRIVE_FOLDER` and `GIT_TAG` for each version (for example `sched_data` + `data-v1`, then `sched_data_v2` + `data-v2`). Never mix files from different versions in one folder.

Run the cells from top to bottom (Runtime > Run all).

In [ ]:
# ---- EDIT THESE ----
YOUR_NAME    = "p1"      # used in the file name
SEED_START   = 0         # p1: 0, p2: 10000, p3: 20000, final test set: 900000
PER_FAMILY   = 2000      # workloads per family (6 families). Final test set: 100
REPO_URL     = "https://github.com/yugarora2/AI_cpu_scheduler.git"
GIT_TAG      = "data-v1" # frozen code version everyone uses ("" = latest main)
DRIVE_FOLDER = "sched_data"   # a second dataset version needs its own folder, e.g. sched_data_v2

## Set up (clone code, install, run the tests)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/proj'):
    !git clone -q {REPO_URL} /content/proj
%cd /content/proj
if GIT_TAG:
    !git checkout -q {GIT_TAG}
!git log --oneline -1
!pip install -q -r requirements.txt

# No GitHub? Upload the project zip to Drive, then use these two lines instead of the clone above:
# !unzip -q /content/drive/MyDrive/adaptive_scheduler.zip -d /content/
# !mv /content/adaptive_scheduler /content/proj
!python test_schedulers.py

## Generate the data

In [ ]:
os.makedirs(f"/content/drive/MyDrive/{DRIVE_FOLDER}", exist_ok=True)
OUT = f"/content/drive/MyDrive/{DRIVE_FOLDER}/part_{YOUR_NAME}.csv"
!python generate_data.py --seed-start {SEED_START} --per-family {PER_FAMILY} --workers {os.cpu_count()} --out {OUT}

## Check the file

In [ ]:
import pandas as pd
d = pd.read_csv(OUT)
print(d.shape, f"| {os.path.getsize(OUT) / 1e6:.1f} MB")
print(d.family.value_counts().to_string())
print("seeds", d.seed.min(), "to", d.seed.max(), "| sim_version", set(d.sim_version))
print("\nSaved to Google Drive:", OUT)
print("Share this file with the person who trains the model.")